# Fase 02: Modelado y API (FastAPI)

Entrenamiento de los modelos XGBoost para TMAX y TMIN (basado en el trabajo de Tamara) y base de la API FastAPI.

## 1. Entrenamiento del Modelo XGBoost

In [ ]:
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import os
import pickle

def entrenar_modelo_tmax(df, idema):
    """Entrena un modelo XGBoost para predecir temperatura máxima basado en temperatura media."""
    # Nos quedamos con los registros de la estación
    df_estacion = df[df['indicativo'] == idema].dropna(subset=['tmed', 'tmax'])
    
    if df_estacion.empty:
        print(f"No hay suficientes datos limpios para {idema}")
        return None
        
    X = df_estacion[['tmed']]
    y = df_estacion['tmax']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Modelo XGBRegressor
    modelo = xgb.XGBRegressor(random_state=42, n_estimators=200, learning_rate=0.01)
    modelo.fit(X_train, y_train)
    
    predicciones = modelo.predict(X_test)
    error = root_mean_squared_error(y_test, predicciones)
    print(f"Error medio del modelo TMAX para {idema}: {error:.2f} ºC")
    
    # Guardamos el modelo en local
    os.makedirs('modelos_max', exist_ok=True)
    with open(f'modelos_max/modelo_{idema}_max.pkl', 'wb') as f:
        pickle.dump(modelo, f)
        
    return modelo

## 2. Implementación de los Endpoints (FastAPI)

Para ejecutar la API usarías en consola: `uvicorn web_api:app --reload`. 
Aquí dejamos la estructura base extraída de `Miguel/api_aemet.py`.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="AEMET", description="API de AEMET para modelos")

class InputFeatures(BaseModel):
    features: list

@app.post("/modelos_max/{idema}/predict")
def prediccion_temp_max(idema: str, input_data: InputFeatures):
    """Simula una predicción de temperatura máxima con el input recibido."""
    # En un entorno real, cargaríamos el .pkl aquí
    valor_simulado = input_data.features[0] * 1.5 + 5.0
    return {
        "status": "success",
        "idema": idema,
        "mensaje": "Predicción de temperatura máxima calculada",
        "prediccion": [[valor_simulado]]
    }